In [ ]:
import pandas as pd

#### Nutze ";" als Trennzeichen: 
data_a = pd.read_csv('churn_part_a.csv', delimiter = ';')

In [ ]:
data_a = pd.read_csv('churn_part_a.csv')

In [ ]:
data_a.head()

In [ ]:
data_b = pd.read_csv('churn_part_b.csv')

In [ ]:
data_b.head()

### Combine Datasets
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html

In [ ]:
data = pd.merge(data_a, data_b, how='inner', on='CustomerId')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data_a.shape

In [ ]:
data_b.shape

### %-Anteil der Kunden die uns verlassen haben

In [ ]:
data["Exited"].sum()/data.shape[0]

### Ermittle Anzahl vollständiger Datensätze

In [ ]:
data.dropna(axis=0).shape

### "Entferne"  Zeilen, in denen entweder Gender oder Age oder Beides NaN ist

In [ ]:
data.dropna(axis=0, subset=['Gender', 'Age']).shape

In [ ]:
data.shape

In [ ]:
### Erstelle Datensatz in dem die Zeilen mit NaNs entfernt wurden

In [ ]:
data_cleaned = data.dropna(axis=0)

In [ ]:
data_cleaned.shape

### Visuelle Analyse

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(data_cleaned['CreditScore'], data_cleaned['Exited'])

In [ ]:
data_cleaned.groupby('Exited')['CreditScore'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].min()

### 1. Vorhersagemodel: Logistische Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

### Selektiere Features (Variablen die für die Vorhersage genutzt werden)

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance']]

In [ ]:
random_variable = 1234

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

### Zeige Koeffizienten formatiert an

In [ ]:
import numpy as np

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Berechne Wahrscheinlichkeit dass uns ein Kunden verlässt (Exited = 1)

In [ ]:
log_reg_simple.predict_proba(x)

jeder Kunde erhählt eine Wahrscheinlichkeit dass er bei uns bleibt (Exited=0) und eine dass er uns verlässt (Exited=1)

### Vorhersage ob Kunden uns verlässt (1) oder bei uns bleibt (0), jede Zahl repräsentiert einen Kunden


In [ ]:
log_reg_simple.predict(x)

### Berechnung Confusion Matrix

In [ ]:
predictions = log_reg_simple.predict(x)

confusion_matrix(beboachtete Werte, vorhergesagte Werte, Sortierung)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_cleaned['Exited'], 
                    predictions, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
accuracy = (113 + 7779)/(113 + 1923 + 181 + 7779)

In [ ]:
accuracy

In [ ]:
precision = 113/(113 + 181)

In [ ]:
precision

### Model with more variables

In [ ]:
data_cleaned.head()

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance', 'EstimatedSalary']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

In [ ]:
predictions = log_reg_simple.predict(x)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_cleaned['Exited'], 
                    predictions, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
112/(112+ 176)

### Berechnung Accuracy mit implementierter Lösung

In [ ]:
metrics.accuracy_score(data_cleaned['Exited'], 
                    predictions)

### Berechnung Precision mit implementierter Lösung

In [ ]:
metrics.precision_score(data_cleaned['Exited'], 
                    predictions)

### Erzeugen von Spalte die den Wert True bei Frauen animmt und False bei Männern

In [ ]:
data_cleaned['Gender_Boolean'] = data['Gender'] == 'Female'

In [ ]:
data_cleaned.head()

### Umwandeln in Zahl

In [ ]:
data_cleaned['Gender_numeric'] = data_cleaned['Gender_Boolean'].astype(int)

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned['Geography'].unique()

### Generierung Dummy Encoding für die Spalte Geography

In [ ]:
geography_dummies = pd.get_dummies(data_cleaned['Geography'], prefix="g")

In [ ]:
geography_dummies

### Mergen von data_cleaned und geography_dummies (Spalten zu data_cleaned hinzfügen)

In [ ]:
data_full = pd.concat([data_cleaned, geography_dummies], axis=1)

In [ ]:
data_full

### Neues Modell mit Herkunftsland und Geschlecht

In [ ]:
features = ['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']

In [ ]:
x_full = data_full[['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x_full, data_full['Exited'])

In [ ]:
predictions_full = log_reg_simple.predict(x_full)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_full['Exited'], 
                    predictions_full, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_full)

In [ ]:
metrics.precision_score(data_full['Exited'], 
                    predictions_full)

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Decision Tree

In [ ]:
from sklearn import tree

Entscheidungsbaum mit 3 Leveln (Depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=3)

Entscheidungsbaum mit 20 Leveln (Depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=20)

### Baum auf Daten anpassen (Trainieren)

In [ ]:
clf_tree.fit(x_full, data_full['Exited'])

### Erzeugen von Vorhersagen

In [ ]:
predictions_tree = clf_tree.predict(x_full)

In [ ]:
cnf_matrix_tree = metrics.confusion_matrix(
                    data_full['Exited'], 
                    predictions_tree, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix_tree

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_tree)

In [ ]:
metrics.precision_score(data_full['Exited'], 
                    predictions_tree)

### Entscheigunsbaum darstellen

In [ ]:
tree.plot_tree(clf_tree, feature_names=x_full.columns)

Installation von Graphviz um optisch schöneren Entscheidungsbaum zu generieren, Installer von hier runterladen:
https://elearning.ohmportal.de/mod/url/view.php?id=511080

im zweiten Schritt für Python installieren


In [ ]:
# !pip install graphviz

In [ ]:
import graphviz

In [ ]:
dot_data = tree.export_graphviz(clf_tree, feature_names=x_full.columns, filled=True, out_file=None)

In [ ]:
graph = graphviz.Source(dot_data)

In [ ]:
graph

In [ ]:
metrics.accuracy_score(data_full['Exited'], 
                    predictions_tree)

- sehr hohe Accuracy, aber Baum passt sich sehr auf die Daten (Overfitting)
- Baum wird bei neuen Daten nicht mehr so gut funktionen

### Random Forest
Rechne verschiedene Bäume und nutze die durchschnittliche Wahrscheinlichtkeit als Vorhersage

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### Random Forest für den wir festlegen, dass in jedem Blatt/Gruppe mindestens 20 Beobachtungen sein müssen (parameter: min_samples_leaf)

In [ ]:
rf = RandomForestClassifier(random_state = random_variable, min_samples_leaf=20)

In [ ]:
rf.fit(x_full, data_full['Exited'])

In [ ]:
predictions_rf = rf.predict(x_full)

In [ ]:
cnf_rf = metrics.confusion_matrix(data_full['Exited'], predictions_rf, labels=[True, False])

In [ ]:
cnf_rf

## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
gbm = GradientBoostingClassifier(random_state = random_variable, n_estimators=100)

In [ ]:
gbm.fit(x_full, data_full['Exited'])

In [ ]:
predictions_gbm = gbm.predict(x_full)

In [ ]:
cnf_gbm = metrics.confusion_matrix(data_full['Exited'], predictions_gbm, labels=[True, False])

# Explainable AI - Shapley Values

In [ ]:
import shap

### need to activate javascript for force plots

In [ ]:
shap.initjs()

## Random Forest

In [ ]:
explainer_rf = shap.TreeExplainer(rf, model_output='probability', data=x_full[features].astype(int).values)

toDo: reduce number of observations for calculating shapley values

In [ ]:
shap_values_rf = explainer_rf.shap_values(x_full[features])

<span style="color:red">
Apparently for random forest, the prediciton function of sklearn generates two values for 0 and 1, we have to decide which to use, shap_values[1], means that we use the likelihood of predicting 1
</span>

In [ ]:
shap.summary_plot(shap_values_rf[1], x_full[features])

<span style="color:red">
The return of the predict function in sklearn for random forest is different to gbm, therefore we need
    to select the right values from the return array:
    <li>
        explainer_rf.expected_value[1]
    </li>
    <li>
    shap_values_rf[1][2]    
    </li>

    Here the explanation for the third observation is provided
</span>

In [ ]:
shap.force_plot(explainer_rf.expected_value[1], shap_values_rf[1][2], x_full.iloc[2,:][features])

## Gradient Boosting

In [ ]:
explainer = shap.TreeExplainer(gbm, model_output='probability', data=x_full[features].astype(int).values)

### Get Average prediction from shapley

In [ ]:
explainer.expected_value

In [ ]:
shap_values = explainer.shap_values(x_full[features])

In [ ]:
shap.summary_plot(shap_values, x_full[features])

The force plot shows why the prediciton for the specific observations differes from the expected value

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[2], x_full.iloc[2,:][features])

In [ ]:
shap_object = shap.Explanation(
        base_values =   explainer.expected_value,
        values = shap_values[2],
        feature_names = features,
        data = shap_values[0].data)

shap.plots.waterfall(shap_object)